In [124]:
import pandas as pd
import numpy as np

In [125]:
%store -r

In [126]:
bm_raw = dfs['bm_raw_new'].copy()
bss_pca = dfs['bss_pca_raw'].copy()
pca_sellers = dfs['PCA for Sellers'].copy()
classification = dfs['classification_mapping'].copy()
Brand_mapping = dfs['Brand_mapping'].copy()
search_roin = dfs['Self serve dashboard'].copy()

In [127]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id', 'brand',
       'commodity_name', 'analytic_vertical', 'analytic_category',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'HL_BU_New', 'Alpha_MP', 'BMP_Tag', 'Team',
       'budget_type', 'cost_model', 'pacing', 'campaign_type', 'campaign_name',
       'status', 'start_date', 'end_date', 'commodity_id', 'Channel',
       'allocated_budget', 'gross_budget', 'uniqueviewcount', 'actioncount',
       'total_cost_x', 'addtobasketcount', 'viewcount', 'cnt_lid',
       'attr_units', 'attr_revenue', 'overburn_share', 'Actual_spend',
       'Budget_adgroup', 'ROI', 'CTR', 'CVR'],
      dtype='object')

In [128]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [129]:
# bss_pca.columns

In [130]:
#  pca_sellers.columns

In [131]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [132]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

In [133]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [134]:
bm_raw['New Alpha/MP'].value_counts()

New Alpha/MP
MP       755466
Alpha     63432
IC           10
Name: count, dtype: int64

In [135]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [136]:
bm_raw['FCFF Alpha/MP'].value_counts()

FCFF Alpha/MP
MP       755465
Alpha     52482
HL        10961
Name: count, dtype: int64

In [137]:
brand_map = dict(zip(Brand_mapping['Vertical'].drop_duplicates(), 
                     Brand_mapping['New SC']))

sc_map = dict(zip(classification['Wrong Tagging'].drop_duplicates(), 
                  classification['New Tag']))

bm_raw = bm_raw.reset_index(drop=True)

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [138]:
# print(bm_raw['New Supercat'].value_counts().to_string())


In [139]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [140]:
bm_raw['SC match FCFF'].value_counts()

SC match FCFF
True     818287
False       621
Name: count, dtype: int64

In [141]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [142]:
print(bm_raw['New BU'].isnull().sum())

621


In [143]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [144]:
bm_raw['Spends'].sum()

np.float64(3500958317.0)

In [145]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


In [146]:
len(bm_raw[bm_raw['Brand'] == "0"])


417

In [147]:
# bm_raw['Brand'].value_counts(normalize=True)

In [148]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id', 'brand',
       'commodity_name', 'analytic_vertical', 'analytic_category',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'HL_BU_New', 'Alpha_MP', 'BMP_Tag', 'Team',
       'budget_type', 'cost_model', 'pacing', 'campaign_type', 'campaign_name',
       'status', 'start_date', 'end_date', 'commodity_id', 'Channel',
       'allocated_budget', 'gross_budget', 'uniqueviewcount', 'actioncount',
       'total_cost_x', 'addtobasketcount', 'viewcount', 'cnt_lid',
       'attr_units', 'attr_revenue', 'overburn_share', 'Actual_spend',
       'Budget_adgroup', 'ROI', 'CTR', 'CVR', 'New Alpha/MP', 'FCFF Alpha/MP',
       'New Supercat', 'SC match FCFF', 'New BU', 'Spends', 'lookup_key',
       'Brand'],
      dtype='object')

# BSS PCA

In [149]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'status',
       'Ad Account ID', 'Ad Account Name', 'Business Account ID',
       'Business Account Name', 'Team', 'Alpha_MP', 'BMP_Tag', 'Channel',
       'lifestyle_supercategory', 'allocated_budget', 'spend', 'views',
       'clicks', 'ppv', 'click_direct_units', 'click_indirect_units',
       'click_direct_revenue', 'click_indirect_revenue', 'Product',
       'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [150]:
# bss_pca = bss_pca[bss_pca['spend']>5].copy()
bss_pca = bss_pca.copy()

In [151]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


In [152]:
bss_pca['New Alpha/MP'].value_counts()

New Alpha/MP
Alpha    438369
MP       215985
IC          758
Name: count, dtype: int64

In [153]:
# class_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

# conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
# choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

# bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])

df_class_unique = classification.drop_duplicates(subset=['Wrong Tagging'])
mapping_dict = dict(zip(df_class_unique['Wrong Tagging'], df_class_unique['New Tag']))

conditions = [
    (bss_pca['BU'] == "Grocery"),
    (bss_pca['BU'] == "Mobile"),
    (bss_pca['Supercategory'] == "PetCare")
]
choices = ["Grocery", "Mobile", "Pets"]
supercat = bss_pca['Supercategory'].map(mapping_dict).fillna(bss_pca['Supercategory'])

bss_pca['New Supercat'] = np.select(conditions, choices, default=supercat)


In [154]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

In [155]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [156]:
bss_pca['SC match FCFF'].value_counts()

SC match FCFF
True     639422
False     15690
Name: count, dtype: int64

In [157]:
bss_pca[bss_pca['SC match FCFF'] == False]['New Supercat'].value_counts()

New Supercat
Not Assigned    15690
Name: count, dtype: int64

In [158]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [159]:
print(bss_pca['New BU'].isnull().sum())

15690


In [160]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

In [161]:
len(bss_pca[bss_pca['Brand'] == "0"])

0

In [162]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [163]:
bss_pca['BrandxBU'] = bss_pca['Brand'].astype(str) + bss_pca['BU'].astype(str)
lookup_map = bss_pca.drop_duplicates('BrandxBU').set_index('BrandxBU')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BrandxBU'].map(lookup_map)

In [164]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,np.where(bss_pca['Check'] == "Not Assigned", "Not Assigned",bss_pca['Check']),bss_pca['New Supercat'])

In [165]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [166]:
print(round(bss_pca[bss_pca['SC match FCFF'] == False ]['spend'].sum(),1))

183867.4


# Search ROINS

In [167]:
search_roin.head()

,PURCHASE_ORDER_ID,PURCHASE_ORDER_NAME,AMOUNT,START_DATE,END_DATE,PAYMENT_CYCLE,ACCOUNT_ID,ACCOUNT_NAME,BUSINESS_ACCOUNT_ID,BUSINESS_ACCOUNT_NAME,CREATED_AT,UPDATED_AT
0,03FGHXVCRETJ,RO/DI/FLIPKA/2578,71400000.0,2026-04-08,2026-04-30,60,QEG6IZGW5JEM,Flipkart Private Brands,JSLFI693SSUH,HIVEMINDS INNOVATIVE MARKET SOLUTIONS PRIVATE ...,2026-04-09T09:03:04.000Z,2026-04-09T09:03:04.000Z
1,04AKUFVAKAQU,Ex/MIN/1P/50338_DHAMPURE,100000.0,2026-04-01,NaN,30,7DYDBPU7TMR7,Dhampur Bio organics Limited,PZH5RYO1JUU5,Dhampur Bio organics Limited,2026-04-02T06:40:50.000Z,2026-04-02T06:40:50.000Z
2,054UJ63VYYWQ,Ex/Li/1P/Nit/0049962,370380.0,2026-04-01,2026-04-30,30,CBRQP8GPMBBD,Fossil_Emporio Armani_PLA_CPC,org-E2OH2JDXW0,Fossil India Private Limited,2026-04-01T12:38:36.000Z,2026-04-01T12:38:36.000Z
3,05DA1O1BK2YP,FKRO/Ele/ITP/A/Nit/02042026,1190000.0,2026-04-02,NaN,30,GJS44AHT7YF0,IT Accessories,EY4AP5YEKF72,Portronics Digital Pvt Ltd,2026-04-03T12:58:13.000Z,2026-04-03T12:58:13.000Z
4,05GTRZNIKINT,RO/DI/FLIPKA/2603,110000.0,2026-04-01,2026-04-30,60,ZP96OYJVD3GP,TVS Motor Company,HFTQ7ZLNB6,Madison Communication Pvt. LTD,2026-04-19T05:22:29.000Z,2026-04-19T05:22:29.000Z


- Start and End of Current Month

In [168]:
search_ro = search_roin[(search_roin['START_DATE'] >= SelfServe_SOM ) & (search_roin['START_DATE'] <= SelfServe_EOM) ].copy()

In [169]:
# search_ro.head()

In [170]:
bm_raw_ro = bm_raw[bm_raw['SC match FCFF'] == True].copy()

In [171]:
bm_raw_ro['total_cost_x'].sum()

np.float64(3500428067.4400005)

In [172]:
bss_pca_ro = bss_pca[bss_pca['SC match FCFF'] == True].copy()

In [173]:
bss_pca_ro['spend'].sum()

np.float64(332429377.1499998)

- Alpha/MP

In [174]:
# alpha_mp_map_bm_raw = dict(zip(bm_raw_ro['advertiser_id'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['FCFF Alpha/MP'].to_dict()
# alpha_mp_map_bss_pca = dict(zip(bss_pca_ro['Ad Account ID'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['FCFF Alpha/MP'].to_dict()

search_ro['Alpha_MP'] = search_ro['ACCOUNT_ID'].map(alpha_mp_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(alpha_mp_map_bss_pca))

In [175]:
print(round(search_ro.groupby('Alpha_MP')['AMOUNT'].sum()))

Alpha_MP
Alpha    1.315188e+09
HL       1.185181e+08
MP       7.740908e+08
Name: AMOUNT, dtype: float64


- BU

In [176]:
bu_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New BU'].to_dict()
bu_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New BU'].to_dict()

search_ro['New BU'] = search_ro['ACCOUNT_ID'].map(bu_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(bu_map_bss_pca))

In [177]:
search_ro['New BU'].value_counts()

New BU
BGM            624
Lifestyle      217
Large          202
Grocery        189
Electronics    175
Home            49
Furniture       14
Mobile           7
Name: count, dtype: int64

- Brand


In [178]:
brand_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['brand'].to_dict()
brand_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['Brands'].to_dict()

search_ro['New Brand'] = search_ro['ACCOUNT_ID'].map(brand_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(brand_map_bss_pca))

- Super Category

In [179]:
sc_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New Supercat'].to_dict()
sc_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New Supercat'].to_dict()

search_ro['New Supercategory'] = search_ro['ACCOUNT_ID'].map(sc_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(sc_map_bss_pca))

In [180]:
# search_ro['New Supercategory'].value_counts()